# ClickCheck

**Phishing + digital safety toolkit**

IBM SkillsBuild National Cybersecurity Bootcamp (Oct 2026)  
BharatCares (CSRBOX Group)

**Rahul** · BCA 3rd Year · Guru Nanak Dev University College, Jalandhar

This notebook is the same project as `clickcheck.py`. Run the cells from top to bottom. Keep `clickcheck.py` in this folder.

I am not attacking any website here. All sample links are dummy examples I typed myself.


## 1. What this project is

During the bootcamp we had 5 masterclasses. I tried to put a bit of each class into one small tool.

| Class | Date | What I used |
|---|---|---|
| Orientation | 5 Oct | program rules, how submission works |
| Masterclass 1 – Cybersecurity 101 | 7 Oct | CIA triad, password hygiene, staying safe |
| Masterclass 2 – Think Before You Click | 10 Oct | phishing, scams, social engineering |
| Masterclass 3 – Hacking 101 | 14 Oct | how an attack moves (simple kill chain) |
| Masterclass 4 – Ethical hacking and forensics | 16 Oct | SHA-256, tampering |
| Masterclass 5 – IBM Bob | 17 Oct | I asked Bob one doubt about urlparse |
| Industry session | 19 Oct | why UPI / SMS scams matter in India |

Main idea: paste a URL or an SMS, get a **LOW / MEDIUM / HIGH** warning with a short reason in plain English.


## 2. Setup

If the next cell fails, it means `clickcheck.py` is not in the same folder as this notebook.

In [1]:
import clickcheck as cc
print("ClickCheck loaded from:", cc.__file__)
print("Python functions:", [x for x in dir(cc) if x in (
    "analyse_url", "analyse_message", "check_password",
    "evidence_demo", "run_tests", "make_incident", "save_report"
)])

ClickCheck loaded from: /home/user/ClickCheck/clickcheck.py
Python functions: ['analyse_url', 'analyse_message', 'check_password', 'evidence_demo', 'run_tests', 'make_incident', 'save_report']


## 3. Password hygiene (Masterclass 1)

We were told not to use `123456` and to prefer a long passphrase. This cell only comments on a string. **Do not type your real password here.**

In [2]:
samples = ["123456", "Rahul@2024", "GreenTea-Jalandhar-Rain-2019!"]
for pw in samples:
    r = cc.check_password(pw)
    print(pw, "->", r["level"], "|", r["summary"])
    for f in r["findings"]:
        print("   ", f["code"], f["detail"])
    print()

123456 -> WEAK | Password looks WEAK (hygiene score 0/57).
    LENGTH_BAD Only 6 characters
    CHARSET Character types used: 1/4
    COMMON This is a very common password
    SEQUENCE Has a simple sequence like 1234/abcd

Rahul@2024 -> FAIR | Password looks FAIR (hygiene score 44/57).
    LENGTH_SHORT Length is 10. 12+ is safer
    CHARSET Character types used: 4/4

GreenTea-Jalandhar-Rain-2019! -> STRONG | Password looks STRONG (hygiene score 57/57).
    LENGTH_OK Length is 29 (12+ is good)
    CHARSET Character types used: 4/4



## 4. URL check (Masterclass 2)

Things I look for (the same tricks they showed in class):

- http instead of https
- raw IP instead of a name
- `@` hiding the real host
- bit.ly and other shorteners
- cheap endings like `.xyz` `.top` `.click`
- brand name sitting on a fake domain (`sbi-kyc-update.xyz`)
- KYC / login sitting on that fake domain

Real sites like `onlinesbi.sbi` and `incometax.gov.in` should come out LOW.

In [3]:
urls = [
    "https://onlinesbi.sbi",
    "https://incometax.gov.in",
    "http://sbi-kyc-update.xyz/login/verify",
    "http://192.168.5.21/hdfc/secure",
    "https://paytm-refund.top/claim",
    "https://bit.ly/3fakelink",
    "http://pspcl-bill-pay.click/kyc",
    "https://amazon.in.account-hold.icu/package",
]
for u in urls:
    r = cc.analyse_url(u)
    print(r["level"].ljust(6), str(r["score"]).rjust(3), u)
    for f in r["findings"]:
        print("         -", f["code"] + ":", f["detail"])
    print()

LOW      0 https://onlinesbi.sbi

LOW      0 https://incometax.gov.in

HIGH    75 http://sbi-kyc-update.xyz/login/verify
         - NO_HTTPS: Uses http, not https
         - ODD_TLD: Unusual domain ending .xyz
         - MANY_HYPHENS: Domain has many hyphens: sbi-kyc-update.xyz
         - BRAND_LOOKALIKE: Looks like it is copying: sbi, kyc (host is sbi-kyc-update.xyz)
         - BAIT_PATH: Path talks about: verify, login

HIGH    47 http://192.168.5.21/hdfc/secure
         - NO_HTTPS: Uses http, not https
         - IP_HOST: Website address is a raw IP: 192.168.5.21
         - BAIT_PATH: Path talks about: secure

HIGH    45 https://paytm-refund.top/claim
         - ODD_TLD: Unusual domain ending .top
         - BRAND_LOOKALIKE: Looks like it is copying: paytm (host is paytm-refund.top)

LOW     15 https://bit.ly/3fakelink
         - SHORTENER: URL shortener: bit.ly

HIGH    75 http://pspcl-bill-pay.click/kyc
         - NO_HTTPS: Uses http, not https
         - ODD_TLD: Unusual domain e

## 5. SMS / WhatsApp text (Masterclass 2)

I used examples that people in Jalandhar actually get: PSPCL (electricity), SBI KYC, fake iPhone prize, Amazon warehouse, and a work-from-home job. The last normal message about class timing should stay LOW.

In [4]:
msgs = [
    "Dear Customer, your PSPCL electricity connection will be DISCONNECTED in 24 hours. Update KYC immediately: http://pspcl-kyc-update.xyz/verify",
    "SBI: Your account will be blocked today. Click here to complete KYC https://sbi-secure-kyc.top/login and share OTP with us.",
    "CONGRATULATIONS you have won a free iPhone 16. Pay Rs 499 processing fee on UPI and send screenshot. Do not share this with anyone.",
    "Your Amazon order is held at Jalandhar warehouse. Pay Rs 37 extra charges. Scan this QR / send money to 9876543210. Collect request sent.",
    "Hi Rahul, class is at 10 am tomorrow in room 12. Please bring the practical file.",
    "Part time data entry job, work from home, 15000 per week. Pay 999 registration on this UPI id then we will add you on WhatsApp only.",
]
for i, m in enumerate(msgs, 1):
    r = cc.analyse_message(m)
    print("MSG", i, r["level"], "score", r["score"])
    print(" ", m[:90] + ("..." if len(m) > 90 else ""))
    for f in r["findings"]:
        print("   *", f["code"] + ":", f["detail"])
    print("  advice:", r["findings"][0]["advice"] if r["findings"] else "looks ok")
    print()

MSG 1 HIGH score 61
  Dear Customer, your PSPCL electricity connection will be DISCONNECTED in 24 hours. Update ...
   * URGENCY: Tries to create panic / hurry [immediately, will be disconnected]
   * THREAT: Uses fear (block, cut, legal, KYC) [disconnected]
   * GENERIC: Starts with Dear Customer / Dear User (no name)
   * BAD_LINK_IN_MSG: Message contains a risky link: http://pspcl-kyc-update.xyz/verify (HIGH)
  advice: Scammers do not give you time to think. Pause and call the company on the number from their official site.

MSG 2 HIGH score 88
  SBI: Your account will be blocked today. Click here to complete KYC https://sbi-secure-kyc...
   * URGENCY: Tries to create panic / hurry [account will be blocked]
   * THREAT: Uses fear (block, cut, legal, KYC) [blocked]
   * SENSITIVE: Asks for OTP / PIN / Aadhaar / PAN / CVV [share otp]
   * CLICK: Pushes you to click / verify / download [click here]
   * BAD_LINK_IN_MSG: Message contains a risky link: https://sbi-secure-kyc.top/login (H

## 6. CIA triad and kill chain (Masterclass 3)

Just printing the mapping for the PSPCL sample so it is easy to see in the notebook.

In [5]:
r = cc.analyse_message(
    "Dear Customer, your PSPCL electricity connection will be DISCONNECTED in 24 hours. "
    "Update KYC immediately: http://pspcl-kyc-update.xyz/verify"
)
print(r["summary"])
print("CIA:")
for x in r["cia"]:
    print(" -", x)
print("Kill chain:")
for x in r["kill_chain"]:
    print(" -", x)

Message risk HIGH (61 points). 4 warning(s).
CIA:
 - Availability (they threaten to cut service / lock account)
Kill chain:
 - Delivery (SMS / WhatsApp / email)


## 7. Tiny forensics demo (Masterclass 4)

They showed that a hash is like a fingerprint. I write a dummy bill file, hash it, add one extra line, hash it again. If the two SHA-256 values differ, the file was changed.

In [6]:
ev = cc.evidence_demo("output")
print("file        :", ev["file"])
print("hash before :", ev["hash_before"])
print("hash after  :", ev["hash_after"])
print("same?       :", ev["match"])
print(ev["summary"])

file        : output/sample_evidence.txt
hash before : d1a89e10c3300d3677248b8cf325ffcd496419294f37866a1ea576852fca05fb
hash after  : 2683a177d0729c0730f493da3968082b1bacd3f40c42c6f484b8086f377bfe5d
same?       : False
Hashes DO NOT match. Even a small edit changes SHA-256. This is how we notice tampering.


## 8. Save one incident note

This is the PSPCL case as a text + json file. In real life this would be the first page of a complaint, not the complaint itself.

In [7]:
inc = cc.make_incident(
    url_result=cc.analyse_url("http://pspcl-kyc-update.xyz/verify"),
    msg_result=cc.analyse_message(
        "Dear Customer, your PSPCL electricity connection will be DISCONNECTED in 24 hours. "
        "Update KYC immediately: http://pspcl-kyc-update.xyz/verify"
    ),
    extra_note="Demo only. Fake PSPCL KYC SMS, common in Punjab.",
)
txt, js, text = cc.save_report(inc, "output")
print("wrote", txt)
print("wrote", js)
print()
print(text)

wrote output/clickcheck_report.txt
wrote output/clickcheck_report.json

ClickCheck - Incident Note
Generated: 2026-10-26 10:55:51
Student  : Rahul, GNDU College Jalandhar, BCA 3rd Year
Program  : IBM SkillsBuild National Cybersecurity Bootcamp, Oct 2026
------------------------------------------------------------
Overall  : HIGH  (score 75)
Advice   : Do not click, do not share OTP, do not scan any QR. Report the SMS as spam. If money already left, call your bank helpline from the number on the back of the card and also report on cybercrime.gov.in
------------------------------------------------------------

[URL]
  Input : http://pspcl-kyc-update.xyz/verify
  Level : HIGH (75 pts)
  URL risk HIGH (75 points). 5 warning(s) found.
    - [NO_HTTPS] +10  Uses http, not https
      advice: Real banks in India use https. Check the lock icon.
    - [ODD_TLD] +15  Unusual domain ending .xyz
      advice: Govt and banks mostly use .in / .com / .gov.in
    - [MANY_HYPHENS] +8  Domain has many h

## 9. Tests

I wrote 16 small checks so I know I did not break the scoring after changing weights.

In [8]:
passed, total, rows = cc.run_tests()
print()
if passed == total:
    print("All tests passed.")
else:
    print("Some tests failed. Check the list above.")


--- tests ---
PASS - real SBI is not HIGH
PASS - fake SBI xyz is HIGH or MEDIUM
PASS - IP host gets IP_HOST
PASS - shortener flagged
PASS - lookalike brand flagged
PASS - plain example.com stays LOW
PASS - normal college msg is LOW
PASS - OTP + KYC is HIGH
PASS - prize scam at least MEDIUM
PASS - PSPCL fake is HIGH or MEDIUM
PASS - 123456 is WEAK
PASS - long passphrase not WEAK
PASS - tamper changes hash
PASS - hash is 64 hex chars
PASS - @ trick flagged
PASS - UPI collect flagged
Result: 16/16 passed

All tests passed.


## 10. Try your own (optional)

Change the two strings below and run the cell. Use a dummy link, not a live phishing site.

In [ ]:
my_url = "https://example.com"
my_msg = "Hi, meeting is at 4 pm."

print("URL")
cc.print_result(cc.analyse_url(my_url))
print("MESSAGE")
cc.print_result(cc.analyse_message(my_msg))

## 11. What I could not do / next steps

- No live internet blacklist (I did not want API keys).
- Shortener like bit.ly is only flagged, not expanded.
- Cannot say 100% that a site is phishing. It only shows red flags.
- Later I want to add Punjabi language SMS, because a lot of local scam texts are in Punjabi.

If money is already lost: bank helpline (number on the card) and cybercrime.gov.in. This notebook cannot reverse a UPI payment.

---

Rahul · GNDU College, Jalandhar · BCA 3rd Year · October 2026
